# 1957 Perceptron

Frank Rosenblatt's perceptron is one of the earliest neural network models and a useful starting point for an AI timeline notebook.

## High-Level Ideas

A perceptron receives one or more input values and combines them into a single score before making a binary decision.

- **Weights** (`weights`) control how strongly each input affects the result. A positive weight pushes the score upward when the input is `1`; a negative weight pushes it downward.
- **Bias** (`bias`) sets the threshold for the decision. Its goal is to make the perceptron more or less strict about when it should return `1`.

The perceptron first calculates the weighted sum:

$$z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b$$

Then it applies a step activation function:

$$\hat{y} = \begin{cases} 1, & z \ge 0 \\ 0, & z < 0 \end{cases}$$

In the code below, `total` is `z`, and `return 1 if total >= 0 else 0` is the step function.

During training, each example has `inputs` and a `desired_output`. The trainer compares the perceptron's prediction with the desired output:

$$error = desired\ output - prediction$$

Then it updates the weights and bias:

$$w_i = w_i + learning\ rate \times error \times x_i$$

$$b = b + learning\ rate \times error$$

## Example: AND and OR Logic Gates

The code below trains two perceptrons from simple truth tables. One learns the `AND` gate and the other learns the `OR` gate, showing how weights and bias turn examples into a binary decision rule.

In [1]:
# A perceptron stores one weight per input plus a bias that shifts the decision boundary.
class Perceptron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def predict(self, inputs):
        # The dot product measures evidence; the bias lets the boundary move away from zero.
        total = sum(weight * value for weight, value in zip(self.weights, inputs)) + self.bias
        # Step activation turns the numeric score into a binary class.
        return 1 if total >= 0 else 0


class Trainer:
    def __init__(self, perceptron, learning_rate=1):
        self.perceptron = perceptron
        self.learning_rate = learning_rate

    def train(self, training_data, epochs=10):
        # One epoch gives the perceptron a chance to learn from every truth-table row.
        for _ in range(epochs):
            mistakes = 0

            for inputs, desired_output in training_data:
                prediction = self.perceptron.predict(inputs)
                # The signed error determines which direction to move the parameters.
                error = desired_output - prediction

                if error != 0:
                    # Only mistaken examples update weights; active inputs receive the correction.
                    self.perceptron.weights = [
                        weight + self.learning_rate * error * value
                        for weight, value in zip(self.perceptron.weights, inputs)
                    ]
                    self.perceptron.bias += self.learning_rate * error
                    mistakes += 1

            # A mistake-free pass means this linearly separable table has converged.
            if mistakes == 0:
                break

        return self.perceptron


# Each pair contains two binary inputs and the desired logic-gate output.
and_training_data = [
    ((0, 0), 0),
    ((0, 1), 0),
    ((1, 0), 0),
    ((1, 1), 1),
]

or_training_data = [
    ((0, 0), 0),
    ((0, 1), 1),
    ((1, 0), 1),
    ((1, 1), 1),
]

# Both models start identically; their different examples produce different learned boundaries.
and_gate = Perceptron(weights=[0, 0], bias=0)
or_gate = Perceptron(weights=[0, 0], bias=0)

# The trailing semicolon suppresses a nondeterministic object representation in notebook output.
Trainer(and_gate).train(and_training_data)
Trainer(or_gate).train(or_training_data);

In [2]:
# Evaluate every possible pair so the learned decision rules are easy to inspect.
binary_inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

print(f"Learned AND parameters: weights={and_gate.weights}, bias={and_gate.bias}")
print("AND gate")
for values in binary_inputs:
    print(f"{values} -> {and_gate.predict(values)}")

print(f"\nLearned OR parameters: weights={or_gate.weights}, bias={or_gate.bias}")
print("OR gate")
for values in binary_inputs:
    print(f"{values} -> {or_gate.predict(values)}")


# Assertions make the notebook fail during pre-commit if either model regresses.
for inputs, desired_output in and_training_data:
    assert and_gate.predict(inputs) == desired_output

for inputs, desired_output in or_training_data:
    assert or_gate.predict(inputs) == desired_output

print("\nAll perceptron gate checks passed.")

Learned AND parameters: weights=[2, 1], bias=-3
AND gate
(0, 0) -> 0
(0, 1) -> 0
(1, 0) -> 0
(1, 1) -> 1

Learned OR parameters: weights=[1, 1], bias=-1
OR gate
(0, 0) -> 0
(0, 1) -> 1
(1, 0) -> 1
(1, 1) -> 1

All perceptron gate checks passed.


## Why This Mattered

The perceptron showed that a machine could learn a decision rule from examples instead of being programmed with every rule by hand. Even though single-layer perceptrons have important limits, the idea of adjustable weights became a foundation for later neural networks and modern deep learning.